### Google IP

In [ ]:
dennie@X390:~/linux-tut$ dig @8.8.8.8 google.com

; <<>> DiG 9.18.39-0ubuntu0.24.04.7-Ubuntu <<>> @8.8.8.8 google.com
; (1 server found)
;; global options: +cmd
;; Got answer:
;; ->>HEADER<<- opcode: QUERY, status: NOERROR, id: 60426
;; flags: qr rd ra; QUERY: 1, ANSWER: 1, AUTHORITY: 0, ADDITIONAL: 1

;; OPT PSEUDOSECTION:
; EDNS: version: 0, flags:; udp: 512
;; QUESTION SECTION:
;google.com.                    IN      A

;; ANSWER SECTION:
google.com.             300     IN      A       172.217.19.14

;; Query time: 55 msec
;; SERVER: 8.8.8.8#53(8.8.8.8) (UDP)
;; WHEN: Tue Sep 22 20:26:30 CEST 2026
;; MSG SIZE  rcvd: 55

dennie@X390:~/linux-tut$ nslookup google.com
Server:         10.255.255.254
Address:        10.255.255.254#53

Non-authoritative answer:
Name:   google.com
Address: 142.251.141.174
Name:   google.com
Address: 2a00:1450:4014:801::200e

Nice work running those commands! Let me explain what's happening—this is actually a great learning moment.

## Why `142.250.184.78` "doesn't work"

Google serves `google.com` from a **huge, rotating pool of IP addresses**, distributed across many data centers worldwide. Any single IP like `142.250.184.78` is just one node in that pool, and it may:

- No longer be assigned to that hostname, or
- Only respond correctly to requests using the right **Host header** (i.e., `Host: google.com`), so a bare IP visit in the browser can behave oddly or fail.

So it's not "wrong"—it's just **time-sensitive and context-dependent**. Google's DNS answers change every few minutes.

## What your output shows

**`dig @8.8.8.8 google.com`** → you asked Google's DNS directly:

In [ ]:
google.com.  300  IN  A  172.217.19.14

- That's the IP Google's own resolver returned _right now_.
- The `300` is the **TTL** (time-to-live) in seconds—after 5 minutes, this answer expires and must be looked up again. That's why the address keeps changing!

**`nslookup google.com`** → used your system's **default resolver** (the router at `10.255.255.254`), which returned:

In [ ]:
142.251.141.174
2a00:1450:4014:801::200e  (IPv6)

Notice the two tools give **different** answers. Both are valid—they're just different nodes in Google's network chosen by different resolvers.

## Key takeaways

|Item|Meaning|
|---|---|
|`8.8.8.8`|Google's **DNS server** (a resolver), not a website IP|
|`142.250.x.x`, `172.217.x.x`, `142.251.x.x`|All are **valid Google web-server IPs** at various times|
|`300` (TTL)|Answers expire; re-query to get fresh ones|
|Different IPs from `dig` vs `nslookup`|Normal—different resolvers, geo-load balancing|

## Try this

In [ ]:
dennie@X390:~/linux-tut$ for i in 1 2 3; do dig +short @8.8.8.8 google.com; done
172.217.19.14
172.217.19.14
172.217.19.14

Interesting result! You got the **same IP three times in a row**—this is expected and actually teaches something important about DNS.

## Why the same IP?

DNS answers are **cached**, and the caching happens in a couple of places:

1. **Google's public resolver (8.8.8.8) caches the answer too.** When your first query comes in, it resolves the name and stores the result for the duration of the **TTL** (remember the `300` = 5 minutes we saw earlier). Subsequent queries within that window get the cached answer instantly—no new lookup to Google's authoritative servers.
    
2. **Round-robin rotation isn't guaranteed per-query.** Google _can_ return multiple A records that the client rotates through, but it doesn't have to hand you a different one on every single request. Often it returns a small set of "best" nodes and keeps serving them until the cache expires.
    

So getting the same answer 3 times in a row is completely normal—it just means the cached entry is still valid.

## What to try instead

To see the rotation, wait for the TTL to expire, or flush your lookup by using **different resolvers** and **different query types**:

In [ ]:
# Ask different resolvers — they each have their own caches
dig +short @1.1.1.1 google.com
dig +short @8.8.8.8 google.com
dig +short @9.9.9.9 google.com

# Or rotate through Google's IPs to see multiple records
dig @8.8.8.8 google.com +noall +answer

You can also query a name that tends to return many addresses:

In [ ]:
dig +short @8.8.8.8 www.google.com
dig +short @8.8.8.8 google.com | wc -l

## The takeaway

|Behavior|Reason|
|---|---|
|Same IP 3×|Cached answer, still within TTL (300 s)|
|Different IPs across resolvers|Each resolver has its own independent cache|
|IPs change over time|TTL expires → fresh lookup → possibly new node|

This is exactly why a fixed IP like `142.250.184.78` "stops working"—the DNS-level answer that pointed a client to it has long since expired, and Google now advertises different nodes.

You're getting a solid feel for how real-world DNS caching and load balancing behave. Want to dig into how to inspect the TTL and watch an entry expire in real time?